Checking for GPU availability

In [2]:
!nvidia-smi
!nvcc --version

Sun Feb  8 14:05:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Step 2: Install llama.cpp Python Bindings

In [ ]:
# Install llama-cpp-python with GPU support
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir --force-reinstall --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 143.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 238.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 378.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 257.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 215.5 MB/s eta 0:00:00


In [ ]:
# Download model from Hugging Face
!pip install huggingface_hub
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
    filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    local_dir="./models"
)

print(f"Model downloaded to: {model_path}")

In [ ]:
from llama_cpp import Llama

# Load model with GPU acceleration
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,      # Offload all layers to GPU
    n_ctx=2048,           # Context window (tokens)
    verbose=True          # Show loading info
)

print("✓ Model loaded successfully")

In [ ]:
import time

# Prepare prompt
prompt = "Explain what a GPU is in one sentence."

# Generate response
start_time = time.time()

response = llm(
    prompt,
    max_tokens=100,       # Max length of response
    temperature=0.7,      # Creativity (0=deterministic, 1=creative)
    stop=["User:", "\n\n"]  # Stop tokens
)

end_time = time.time()

# Extract results
output_text = response['choices'][0]['text']
tokens_generated = response['usage']['completion_tokens']
time_taken = end_time - start_time
tokens_per_sec = tokens_generated / time_taken

# Display results
print(f"Response: {output_text}")
print(f"\n--- Performance ---")
print(f"Tokens generated: {tokens_generated}")
print(f"Time taken: {time_taken:.2f}s")
print(f"Speed: {tokens_per_sec:.1f} tokens/sec")

In [ ]:
# Check VRAM usage
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
def chat(user_message):
    response = llm(
        f"User: {user_message}\nAssistant:",
        max_tokens=200,
        temperature=0.7,
        stop=["User:"]
    )
    return response['choices'][0]['text'].strip()

# Try it
print(chat("What's the capital of France?"))
print(chat("Write a haiku about coding."))

End